In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 25
two_colors = ['#6384B3', '#F08743']

In [ ]:
data = pd.read_pickle('./full_data.pkl')
data = data.loc[data['age']>18]
print(data.shape[0])
data_add = pd.read_pickle('./ini_entropy.pkl')
data_beh = pd.read_pickle('./user_beh.pkl')

data = data.drop(columns=['expose_num_ex_search1', 'attention_span1', 'watched_search1','top10avg'])
data = data.merge(data_add, on='user_id')
data = data.merge(data_beh, on='user_id')

In [ ]:
# The expose_num values are already normalized by five viewing hours.


In [ ]:
def age_range_mapping(x):
    if 18<=x<=23:
        return '18-23'
    elif 24<=x<=30:
        return '24-30'
    elif 31<=x<=40:
        return '31-40'
    elif 41<=x<=50:
        return '41-50'
    elif 51<=x<=60:
        return '51-60'
    elif x>=61:
        return '60+'
    else:
        return None
    
def get_outliers(data, multiplier=1.5):
    # Calculate Q1 (25th percentile) and Q3 (75th percentile).
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)

    # Calculate the interquartile range.
    IQR = Q3 - Q1

    # Define the lower and upper outlier bounds.
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR

    return lower_bound, upper_bound


def bernoulli_ci(values):
    import statsmodels.api as sm
    n = len(values)
    p = values.mean()
    conf_int = sm.stats.proportion_confint(p * n, n, method='normal')
    return conf_int


In [ ]:
data['age_range'] = data['age'].transform(lambda x: age_range_mapping(x))
data['activeness'] = data.apply(lambda x: x['collect1']+x['forward1']+x['comment1'], axis=1)

data = data.dropna()
data = data.loc[data['gender']!='UNKNOWN']
data = data.loc[data['fre_community_level_eng']!='unknown']
data = data.loc[data['fre_city_level_eng']!='unknown']
data = data.loc[data['mod_price_range']!='UNKNOWN']
data = data.loc[data['mod_price_range']!='UNKNOWN']
print(data.shape)


data_orig = data.copy()

s0_min, s0_max = get_outliers(data_orig['watch_entropy_1st'])
print('s0', s0_min, s0_max)
data  = data.loc[data['watch_entropy_1st']>=s0_min]
data  = data.loc[data['watch_entropy_1st']<=s0_max]

activeness_min, activeness_max = get_outliers(data_orig['activeness'])
print('activeness', activeness_min, activeness_max)
data  = data.loc[data['activeness']>=activeness_min]
data  = data.loc[data['activeness']<=activeness_max]


search_min, search_max = get_outliers(data_orig['watched_search1'])
print('search', search_min, search_max)
data  = data.loc[data['watched_search1']>=search_min]
data  = data.loc[data['watched_search1']<=search_max]


attention_min, attention_max = get_outliers(data_orig['attention_span1'])
print('attention', attention_min, attention_max)
data  = data.loc[data['attention_span1']>=attention_min]
data  = data.loc[data['attention_span1']<=attention_max]


top10avg_min,top10avg_max = get_outliers(data_orig['top10avg'])
print('top10avg', top10avg_min,top10avg_max)
data  = data.loc[data['top10avg']>=top10avg_min]
data  = data.loc[data['top10avg']<=top10avg_max]

In [ ]:
data_orig = []
print(data.shape)
data = data[['user_id','m_ratio','gender', 'age' ,'age_range','fre_city_level_eng', 'fre_community_level_eng', 'mod_price_range',
            'activeness', 'watched_search1', 'attention_span1', 'top10avg', 'entertainment_ratio',
            'knowledge_ratio', 'news_ratio','watch_entropy_1st', 'expose_num1']]

In [ ]:
data_ratio = pd.concat([pd.read_pickle('./ratio_for_cleaned_p1.pkl'), pd.read_pickle('./ratio_for_cleaned_p2.pkl')])
data_ratio =  data_ratio.sort_values(by=['user_id', 'five_hour_seq']).groupby('user_id')['ratio'].apply(lambda x: np.array(list(x))).reset_index()

In [ ]:
data_ratio = data_ratio.rename(columns={'ratio':'ratio_seq'})
data = data.merge(data_ratio, on='user_id')
data_ratio = []

In [ ]:
data.columns

In [ ]:
from scipy.stats import t,sem,norm
data_std = np.std(data['m_ratio'])
data_avg = np.mean(data['m_ratio'])

# Tail probability alpha.
alpha_list = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
criteria_list = []
for alpha in alpha_list:
    z_value = norm.ppf(alpha/2)
    print("Entropy bounds:", 1-z_value*data_std, 1+z_value*data_std)
    criteria_list.append(1+z_value*data_std)

In [ ]:
population_list = []
gender_bias = []
age_bias = []
income_bias = []
city_bias = []
length_list = []
depth_list = []

non_length_list = []
non_depth_list = []

for c in criteria_list:
    data['indicator'] = data['m_ratio'].transform(lambda x: 1 if x<c else 0)
    population_list.append(data['indicator'].mean())
    length_list.append(data.loc[data['indicator']==1]['ratio_seq'].transform(lambda x: np.mean(x<1)))
    depth_list.append(data.loc[data['indicator']==1]['ratio_seq'].transform(lambda x: np.mean(x)))
    non_length_list.append(data.loc[data['indicator']==0]['ratio_seq'].transform(lambda x: np.mean(x<1)))
    non_depth_list.append(data.loc[data['indicator']==0]['ratio_seq'].transform(lambda x: np.mean(x)).mean())

    gender_tmp = data.groupby('gender')['indicator'].mean().reset_index()
    age_tmp = data.groupby('age_range')['indicator'].mean().reset_index()
    income_tmp = data.groupby('mod_price_range')['indicator'].mean().reset_index()
    city_tmp = data.groupby('fre_community_level_eng')['indicator'].mean().reset_index()

    gender_bias.append((gender_tmp.loc[gender_tmp['gender']=='F']['indicator'].to_numpy()[0]-gender_tmp.loc[gender_tmp['gender']=='M']['indicator'].to_numpy()[0])/data['indicator'].mean())
    age_bias.append((age_tmp.loc[age_tmp['age_range']=='60+']['indicator'].to_numpy()[0]-age_tmp.loc[age_tmp['age_range']=='18-23']['indicator'].to_numpy()[0])/data['indicator'].mean())
    income_bias.append((income_tmp.loc[income_tmp['mod_price_range']=='1-1000']['indicator'].to_numpy()[0]-income_tmp.loc[income_tmp['mod_price_range']=='5000+']['indicator'].to_numpy()[0])/data['indicator'].mean())   
    city_bias.append((city_tmp.loc[city_tmp['fre_community_level_eng']=='Countryside']['indicator'].to_numpy()[0]-city_tmp.loc[city_tmp['fre_community_level_eng']=='City']['indicator'].to_numpy()[0])/data['indicator'].mean())


In [ ]:
data.columns

In [ ]:
# Estimate the total number of interaction records.
# For each user, multiply expose_num1 by 5 and the sequence length, then sum across users.
total_interactions = (data['expose_num1'] * 5 * data['ratio_seq'].apply(len)).sum()
print(f"Estimated total interaction records: {total_interactions:,.0f}")

In [ ]:
from scipy import interpolate

n, bins = np.histogram(data['m_ratio'], bins=100, density=True)
bins = (bins[1:]+bins[:-1])/2
f = interpolate.interp1d(bins, n, fill_value='extrapolate') 
min_delta_entropy = np.min(data['m_ratio'])
max_delta_entropy = np.max(data['m_ratio'])
sep_bins = [min_delta_entropy]+criteria_list+[max_delta_entropy]

colors = sns.cubehelix_palette(start=.5, rot=-.5, n_colors=17)[:12][::-1]+['#F0F0F0']
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(6.5, 6))
color_cnt = 0
former_str = '0%'
for s, e in zip(sep_bins[:-1], sep_bins[1:]):
    idx = np.where((bins>=s)&(bins<e))
    s_bins = np.linspace(s, e, 100)
    plt.plot(s_bins, f(s_bins), color=colors[color_cnt])
    plt.fill_between(y1=f(s_bins), x=s_bins, color=colors[color_cnt])
    if color_cnt < 4:
        plt.scatter(e, np.max(f(s_bins)), color=colors[color_cnt], marker='x',zorder=20)
        plt.hlines(xmin=min_delta_entropy-0.1, xmax=e, y=np.max(f(s_bins)), color=colors[color_cnt], linestyle='--',zorder=20)
        plt.text(x=min_delta_entropy-0.1, y=np.max(f(s_bins))+0.03, s=r'$\alpha=${}'.format(alpha_list[color_cnt]/2), color=colors[color_cnt], fontsize=20)
        # former_str = f"{int(labels[color_cnt][1]*num)}%"
    # if color_cnt > 5:
    #     plt.scatter(e, np.min(f(s_bins)), color=colors[color_cnt], marker='x')
        # plt.hlines(xmin=min_delta_entropy, xmax=e, y=np.min(f(s_bins)), color=colors[color_cnt], linestyle='--')
        
       
    color_cnt += 1
# plt.ylim(0, 1.75)
plt.xlim(-0.2, 2)
# plt.yticks([])
# plt.yticks([0, 0.5, 1, 1.5])
plt.ylabel(r'$P(\tilde{s}_{\text{min}})$')
plt.xlabel(r'$\tilde{s}_{\text{min}}$')
plt.savefig('./figures/figure1/m_ratio_pdf.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
criteria_list

In [ ]:
data.columns
data[['user_id', 'm_ratio', 'gender', 'age', 'age_range', 'fre_city_level_eng', 'fre_community_level_eng', 'mod_price_range']].to_csv('./cleaned_data_1216.csv')

In [ ]:
np.sum(data['m_ratio']<1)/data.shape[0]

In [ ]:
depth_list

In [ ]:
beta_list =  1 - np.array(alpha_list)/2
severity_list = 1- np.array(criteria_list)
avg_severity_list = 1- np.array([np.mean(d) for d in depth_list])
alpha_list = np.array(alpha_list) 

In [ ]:
plt.figure(figsize=(6.5,6))
plt.plot(beta_list, criteria_list, marker='o', label=r'$\tilde{s}_{\text{min}}$', lw=2, markersize=7, color='#6384B3')
plt.plot(beta_list, np.array([np.mean(d) for d in depth_list]), marker='s', label=r'$\tilde{s}_{\text{avg}}$', lw=2, markersize=7, color='#F08743',ls='--')
plt.vlines(x=0.9,ymin=0.3, ymax=1.2, color='gray', lw=10, alpha=0.2)
plt.ylim(0.38, 1.04)
plt.yticks([0.4, 0.6, 0.8,  1.0])
plt.xticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
plt.xlabel(r'$1-\alpha$')
plt.ylabel(r'$\tilde{s}$')
plt.legend()
plt.savefig('./figures/figure1/depth.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
# plt.plot(beta_list, non_depth_list,marker='s')

In [ ]:
# As in the criteria_list loop, classify users at the threshold and calculate the SD among indicator==1 users.
c_adopted = criteria_list[3]
data['indicator'] = data['m_ratio'].transform(lambda x: 1 if x<c_adopted else 0)
# SD of s_min: the SD of m_ratio among users meeting the criterion.
s_min_std = np.std(data.loc[data['indicator']==1, 'm_ratio'])
# SD of s_avg: the SD of user-level sequence means among classified users, from the matching depth_list column.
s_avg_std = np.std(depth_list[idx_alpha])
print(f'SD of s_min = {s_min_std:.4f}, SD of s_avg = {s_avg_std:.4f}')

In [ ]:
diff = []
for i in range(11):
    df_tmp1 = length_list[i].to_frame()
    df_tmp1['indicator'] = 1
    df_tmp2 = non_length_list[i].to_frame()
    df_tmp2['indicator'] = 0
    df = pd.concat([df_tmp1, df_tmp2])
    diff.append(df.groupby('indicator').mean().to_numpy()[0]-df.groupby('indicator').mean().to_numpy()[1])
print(diff)

In [ ]:
# length 
plt.figure(figsize=(6.5,6))
df_tmp1 = length_list[3].to_frame()
df_tmp1['indicator'] = 1
df_tmp2 = non_length_list[3].to_frame()
df_tmp2['indicator'] = 0
df = pd.concat([df_tmp1, df_tmp2])
sns.barplot(df, x='indicator', y='ratio_seq', order=[1,0], capsize=0.2, width=0.5, palette=two_colors, saturation=1, alpha=1)
plt.xlabel('')
plt.xticks([0,1],['Deep ICs','Others'])
plt.yticks([0, 0.2, 0.4,0.6, 0.8], ['{:.0f}%'.format(100*t) for t in [0, 0.2, 0.4,0.6, 0.8] ])
plt.ylabel(r'Proportion of time in ICs')
plt.savefig('./figures/figure1/length_bars.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

print(df.groupby('indicator')['ratio_seq'].mean())
# Two-sided t test comparing mean ratio_seq values for deep-IC users (indicator=1) and others (indicator=0).
from scipy.stats import ttest_ind
g0 = df.loc[df['indicator']==0, 'ratio_seq']
g1 = df.loc[df['indicator']==1, 'ratio_seq']
t_stat, p_two = ttest_ind(g1, g0, alternative='two-sided')
print(f'Two-sided t-test: t = {t_stat:.4f}, p = {p_two:.4e}')

In [ ]:
plt.figure(figsize=(4,4))
n_1, bins_1 = np.histogram(df.loc[df['indicator']==1]['ratio_seq'], bins=100, density=True)
n_0, bins_0 = np.histogram(df.loc[df['indicator']==0]['ratio_seq'], bins=100, density=True)
bins_0 = (bins_0[1:]+bins_0[:-1])/2
bins_1 = (bins_1[1:]+bins_1[:-1])/2
n_1 = np.cumsum(n_1)/100
n_0 = np.cumsum(n_0)/100
n_0 = np.concatenate([np.array([0]),n_0])
bins_0 = np.concatenate([np.array([0]),bins_0])

n_1 = 1- n_1
n_0 = 1- n_0
plt.plot(bins_1, n_1, lw=2, color='#6384B3')
plt.plot(bins_0, n_0, lw=2, color='#F08743')

print(np.argmin(np.abs(n_1-0.5)))
plt.hlines(xmin=-0.1, xmax=bins_1[89], y=n_1[89], color='r', ls='--')
plt.vlines(ymin=-0.1, x=bins_1[89], ymax=n_1[89], color='r', ls='--')
plt.scatter(x=bins_1[89], y=n_1[89],s=40,  marker='o', color='r', zorder=20)
plt.text(x=bins_1[89]-0.2, y=n_1[89]+0.2, s='{:.2f}%'.format(bins_1[89]*100),fontsize=20)


plt.xticks([0, 0.5, 1], ['0', '50%', '100%'])
plt.xlim(-0.1, 1.1)
plt.ylim(-0.1, 1.1)
plt.xlabel('Proportion of time in ICs')
plt.ylabel('CCDF')
plt.savefig('./figures/figure1/length_ccdf.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
# Plot the distribution of inequality across groups.
plt.figure(figsize=(8,6))
c_adopted = criteria_list[3]
print(c_adopted, population_list[3])
data['indicator'] = data['m_ratio'].transform(lambda x: 1 if x<c_adopted else 0)
# Calculate the risk ratio for the oldest and youngest age groups.
prop_18_23 = data.loc[data['age_range']=='18-23', 'indicator'].mean()
prop_60plus = data.loc[data['age_range']=='60+', 'indicator'].mean()
ratio_60_to_18 = prop_60plus / prop_18_23 if prop_18_23 > 0 else float('nan')
print(f'18-23 proportion: {prop_18_23:.4f}, 60+ proportion: {prop_60plus:.4f}, oldest/youngest risk ratio: {ratio_60_to_18:.2f}')
sns.barplot(data, x='age_range', y='indicator', order=['18-23','24-30','31-40','41-50','51-60','60+'], errorbar=bernoulli_ci, palette=sns.color_palette("Blues", 6), capsize=0.2, edgecolor=sns.color_palette("Blues", 6))
plt.hlines(y=population_list[3], xmin=-0.75, xmax=5.75, lw=5, color='gray', ls='--', alpha=0.5, label='Avg')
plt.ylabel('Proportion of Deep ICs')
plt.xlabel('Age')
plt.xlim(-0.75, 5.75)
plt.legend()
plt.savefig('./figures/figure1/IC_disparity.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)


In [ ]:
data_tmp = data.groupby('age_range')['indicator'].mean().reset_index()
data_tmp = pd.concat([data_tmp, pd.DataFrame({'age_range':['mean'], 'indicator': [data['indicator'].mean()]})])

In [ ]:
data_tmp.to_pickle('./processed_data/inequality_stats.pkl')

In [ ]:
# Compare age inequality as the criterion tightens and selects deeper information cocoons.
plt.figure(figsize=(6.5,6))
plt.plot(beta_list, age_bias, marker='o', lw=2, markersize=7, color='#6384B3')
plt.vlines(x=0.9,ymin=0, ymax=0.85, color='gray', lw=10, alpha=0.2)
# plt.plot(beta_list, age_bias, marker='s', label='Disparity', lw=2, markersize=7, color='#F08743',ls='--')
# plt.plot(beta_list, income_bias, marker='o', lw=2, markersize=7, color='#F08743',ls='--')
# plt.plot(beta_list, gender_bias, marker='o', lw=2, markersize=7, color='#F08743',ls='--')
plt.xticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
plt.ylim(0,0.85)
plt.xlabel(r'$1-\alpha$')
plt.ylabel('Disparity')
plt.savefig('./figures/figure1/vary_alpha.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
# Compare age inequality as the criterion tightens and selects deeper information cocoons.
plt.figure(figsize=(12,6))
plt.plot(beta_list, age_bias, marker='o', lw=2, markersize=7, color='#6384B3',label='Age')
plt.plot(beta_list, income_bias, marker='o', lw=2, markersize=7, color='#984B4B', ls='--', label='Income')
plt.plot(beta_list, gender_bias, marker='o', lw=2, markersize=7, color= '#4F9D9D', ls='--', label='Gender')
plt.xticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
plt.xlabel(r'$1-\alpha$')
plt.ylabel('Disparity')
plt.legend()
plt.savefig('./figures/figure1/vary_alpha_all.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
gender_bias

In [ ]:
# Plot the distribution of inequality across groups.
plt.figure(figsize=(8,3.5))
c_adopted = criteria_list[0]
print(c_adopted, population_list[0])
data['indicator'] = data['m_ratio'].transform(lambda x: 1 if x<c_adopted else 0)
# Calculate the risk ratio for the oldest and youngest age groups.
prop_18_23 = data.loc[data['age_range']=='18-23', 'indicator'].mean()
prop_60plus = data.loc[data['age_range']=='60+', 'indicator'].mean()
ratio_60_to_18 = prop_60plus / prop_18_23 if prop_18_23 > 0 else float('nan')
print(f'18-23 proportion: {prop_18_23:.4f}, 60+ proportion: {prop_60plus:.4f}, oldest/youngest risk ratio: {ratio_60_to_18:.2f}')
sns.barplot(data, x='age_range', y='indicator', order=['18-23','24-30','31-40','41-50','51-60','60+'], errorbar=bernoulli_ci, palette=sns.light_palette("seagreen", 6), capsize=0.2)
plt.hlines(y=population_list[0], xmin=-0.75, xmax=5.75, lw=5, color='gray', ls='--', alpha=0.5, label='Avg')
plt.ylabel('Proportion')
plt.xlabel('Age')
plt.xlim(-0.75, 5.75)
plt.legend()
plt.savefig('./figures/figure1/IC_disparity_most.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
# Compare oldest and youngest proportions; the previous cell defines data['indicator'].
prop_by_age = data.groupby('age_range')['indicator'].mean()
prop_18_23 = prop_by_age['18-23']
prop_60_plus = prop_by_age['60+']
fold = prop_60_plus / prop_18_23
print(f"Age 18-23 proportion: {prop_18_23:.4f}")
print(f"Age 60+ proportion: {prop_60_plus:.4f}")
print(f"Oldest/youngest proportion ratio: {fold:.2f}")

In [ ]:
# Plot the distribution of inequality across groups.
plt.figure(figsize=(8,3.5))
c_adopted = criteria_list[-1]
print(c_adopted, population_list[-1])
data['indicator'] = data['m_ratio'].transform(lambda x: 1 if x<c_adopted else 0)
sns.barplot(data, x='age_range', y='indicator', order=['18-23','24-30','31-40','41-50','51-60','60+'], errorbar=bernoulli_ci, palette=sns.light_palette("orange", 7)[1:], capsize=0.2)
plt.hlines(y=population_list[-1], xmin=-0.75, xmax=5.75, lw=5, color='gray', ls='--', alpha=0.5, label='Avg')
plt.ylabel('Proportion')
plt.xlabel('Age')
plt.xlim(-0.75, 5.75)
plt.legend()
plt.savefig('./figures/figure1/IC_disparity_least.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
def age_range_conti(x):
    if x=='18-23':
        return 0
    elif x=='24-30':
        return 1
    elif x=='31-40':
        return 2
    elif x=='41-50':
        return 3
    elif x=='51-60':
        return 4
    elif x == '60+':
        return 5
    else:
        return None

def income_conti(x):
    if x == '1-1000':
        return 0
    elif x == '1000-2000':
        return 1
    elif x == '2000-4000':
        return 2
    elif x == '4000-5000':
        return 3
    elif x == '5000+':
        return 4
    else:
        return None


data['age_conti'] = data['age_range'].transform(lambda x: age_range_conti(x))
data['mod_price_range_conti'] = data['mod_price_range'].transform(lambda x: income_conti(x))

In [ ]:
data.columns

In [ ]:
dummmy_df = data[['user_id','gender','age_conti', 'fre_city_level_eng', 'fre_community_level_eng', 'mod_price_range_conti',
                  'entertainment_ratio','news_ratio','watch_entropy_1st', 'indicator','m_ratio']]

In [ ]:
dummmy_df = pd.get_dummies(dummmy_df, columns=['gender'],dtype=float)
dummmy_df = pd.get_dummies(dummmy_df, columns=['fre_city_level_eng'],dtype=float)
dummmy_df = pd.get_dummies(dummmy_df, columns=['fre_community_level_eng'],dtype=float)

In [ ]:
dummmy_df = dummmy_df.drop(columns=['gender_M', 'fre_city_level_eng_fifth', 'fre_community_level_eng_City'])

In [ ]:
coefficients_dict = {}
all_coefficients_dict = {}

In [ ]:
import statsmodels.api as sm

X = dummmy_df[['age_conti']]
y = dummmy_df[['indicator']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.Logit(y, X).fit(cov_type='HC0')
print(model.summary())
# Extract coefficients and confidence intervals.
coefficients = model.params
conf = model.conf_int()
conf['coefficients'] = coefficients
conf.columns = ['lower', 'upper', 'coefficients']
coefficients_dict['orig'] = conf.loc['age_conti']
all_coefficients_dict['orig'] = conf


In [ ]:
X = dummmy_df[['age_conti','gender_F']]
y = dummmy_df[['indicator']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.Logit(y, X).fit(cov_type='HC0')
print(model.summary())
# Extract coefficients and confidence intervals.
coefficients = model.params
conf = model.conf_int()
conf['coefficients'] = coefficients
conf.columns = ['lower', 'upper', 'coefficients']
coefficients_dict['gender'] = conf.loc['age_conti']
all_coefficients_dict['gender'] = conf

In [ ]:
dummmy_df.columns

In [ ]:
X = dummmy_df[['age_conti','mod_price_range_conti', 'fre_city_level_eng_first', 'fre_city_level_eng_fourth',
       'fre_city_level_eng_second', 'fre_city_level_eng_third',
       'fre_community_level_eng_Countryside',
       'fre_community_level_eng_County']]
y = dummmy_df[['indicator']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.Logit(y, X).fit(cov_type='HC0')
print(model.summary())
# Extract coefficients and confidence intervals.
coefficients = model.params
conf = model.conf_int()
conf['coefficients'] = coefficients
conf.columns = ['lower', 'upper', 'coefficients']
coefficients_dict['income'] = conf.loc['age_conti']
all_coefficients_dict['income'] = conf

In [ ]:
dummmy_df.columns

In [ ]:
X = dummmy_df[['age_conti', 'entertainment_ratio',
       'news_ratio', 'watch_entropy_1st']]
y = dummmy_df[['indicator']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.Logit(y, X).fit(cov_type='HC0')
print(model.summary())
# Extract coefficients and confidence intervals.
coefficients = model.params
conf = model.conf_int()
conf['coefficients'] = coefficients
conf.columns = ['lower', 'upper', 'coefficients']
coefficients_dict['interests'] = conf.loc['age_conti']
all_coefficients_dict['interests'] = conf

In [ ]:
X = dummmy_df[['age_conti','gender_F',
               'mod_price_range_conti', 'fre_city_level_eng_first', 'fre_city_level_eng_fourth', 'fre_city_level_eng_second', 'fre_city_level_eng_third', 'fre_community_level_eng_Countryside',
               'fre_community_level_eng_County', 'entertainment_ratio', 'news_ratio', 'watch_entropy_1st']]
y = dummmy_df[['indicator']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.Logit(y, X).fit(cov_type='HC0')
print(model.summary())
# Extract coefficients and confidence intervals.
coefficients = model.params
conf = model.conf_int()
conf['coefficients'] = coefficients
conf.columns = ['lower', 'upper', 'coefficients']
coefficients_dict['all'] = conf.loc['age_conti']
all_coefficients_dict['all'] = conf

In [ ]:
dummmy_df['age_conti'].drop_duplicates()

In [ ]:
coefficients_dict

In [ ]:
labels = conf.index
positions = range(len(labels))


# Draw the bar chart.
plt.figure(figsize=(10, 6))
for idx, tt in enumerate(['orig', 'gender', 'income', 'interests', 'all']):
    conf = coefficients_dict[tt]
    plt.errorbar(y=idx, x=np.exp(conf['coefficients']), xerr=[[np.exp(conf['coefficients']) - np.exp(conf['lower'])], [np.exp(conf['upper']) - np.exp(conf['coefficients'])]], capsize=5, elinewidth=2, capthick=2, fmt='o')
    # plt.text(x=1.35, y=idx, s='{:.2f} [{:.2f}, {:.2f}]'.format(np.exp(conf['coefficients']), np.exp(conf['lower']), np.exp(conf['upper'])))

plt.fill_between(x=[0.9,1],y1=[-0.5,-0.5], y2=[4.5, 4.5], color='gray',alpha=0.1)
plt.vlines(x=1, ymin=-0.5, ymax=4.5, color='gray', ls='--')
plt.gca().invert_yaxis()
plt.xlim(0.9,1.5)
plt.ylim(-0.5,4.5)
plt.xlabel('Odds Ratio')
plt.yticks([0,1,2,3,4],[])
plt.savefig('./figures/figure1/odds_ratio.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)



# plt.xticks(positions, labels)
# plt.axhline(0, color='red', linewidth=0.8, linestyle='--')
# plt.title('Coefficients with 95% Confidence Intervals')
# plt.xlabel('Variables')
# plt.ylabel('Effect Size')
# plt.tight_layout()
# plt.show()

In [ ]:
list(filter(lambda x: x not in ['const', 'age_conti'], list(conf.index)))


In [ ]:
dummmy_df['indicator'].mean()

In [ ]:
# ['18-23','24-30','31-40','41-50','51-60','60+'],
labels = conf.index
positions = range(len(labels))
five_colors = sns.color_palette("tab10",5)
x = np.array([i for i in np.linspace(0,1,np.arange(18,24).shape[0]+1)[:-1]] + [1+i for i in np.linspace(0,1,np.arange(24,31).shape[0]+1)[:-1]] \
+[2+i for i in np.linspace(0,1,np.arange(31,41).shape[0]+1)[:-1]] + [3+i for i in np.linspace(0,1,np.arange(41,51).shape[0]+1)[:-1]]\
+[4+i for i in np.linspace(0,1,np.arange(51,61).shape[0]+1)[:-1]] + [5.0])
x_show = np.arange(0,60-18+2)
labels = ['Age', '+Gender', '+Income', '+Interests', '+All']

def get_logit(x, b_0, b_1, others):
    return (1/(1+np.exp(-(b_0+b_1*x+others)))).reshape(1,-1)

def get_y_range(x, conf, dummmy_df):
    b_0 = conf.loc['const']
    b_1 = conf.loc['age_conti']
    other_columns = list(filter(lambda x: x not in ['const', 'age_conti'], list(conf.index)))
    other_values = 0 
    print(other_columns)
    for cc in other_columns:
        mean_values = dummmy_df[cc].mean()
        cc_avg = conf.loc[cc]['coefficients']*mean_values
        other_values += cc_avg
        
    l1 = get_logit(x, b_0['coefficients'], b_1['coefficients'], other_values)
    l2 = get_logit(x, b_0['lower'], b_1['lower'], other_values)
    l3 = get_logit(x, b_0['upper'], b_1['upper'], other_values)
    l4 = get_logit(x, b_0['lower'], b_1['upper'], other_values)
    l5 = get_logit(x, b_0['upper'], b_1['lower'], other_values)
    l_bound = np.vstack([l2, l3, l4, l5])
    l_upper = np.max(l_bound, axis=0,keepdims=True)
    l_lower = np.min(l_bound, axis=0,keepdims=True)
    return l1.reshape(-1), l_upper.reshape(-1), l_lower.reshape(-1)

    

# Draw the bar chart.
plt.figure(figsize=(6.5,6))
for idx, tt in enumerate(['orig', 'gender', 'income', 'interests', 'all']):
    conf = all_coefficients_dict[tt]

    l1, l_upper, l_lower = get_y_range(x, conf, dummmy_df)
    plt.plot(x_show, l1, color=five_colors[idx], lw=2, label=labels[idx])
    plt.fill_between(x=x_show , y1=l_lower, y2=l_upper, color=five_colors[idx], alpha=0.05)
    plt.xticks([i-18 for i in [18, 30, 40, 50, 60]], ['18', '30', '40', '50', '60'])


plt.hlines(y=dummmy_df['indicator'].mean(), xmin=0, xmax=60-18, lw=5, color='gray', ls='--', alpha=0.5,)
plt.legend(fontsize=20)
plt.xlabel('Age')
plt.ylabel('Proprotion of Deep ICs')
plt.savefig('./figures/figure1/odds_ratio_fitted.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
# plt.fill_between(x=[0.9,1],y1=[-0.5,-0.5], y2=[4.5, 4.5], color='gray',alpha=0.1)
# plt.vlines(x=1, ymin=-0.5, ymax=4.5, color='gray', ls='--')
# plt.gca().invert_yaxis()
# # plt.xlim(0.9,1.5)
# # plt.ylim(-0.5,4.5)
# plt.xlabel('Odds Ratio')
# plt.yticks([0,1,2,3,4],[])
# plt.savefig('./figures/figure1/odds_ratio.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

# plt.xticks(positions, labels)
# plt.axhline(0, color='red', linewidth=0.8, linestyle='--')
# plt.title('Coefficients with 95% Confidence Intervals')
# plt.xlabel('Variables')
# plt.ylabel('Effect Size')
# plt.tight_layout()
# plt.show()

In [ ]:
labels = conf.index
positions = range(len(labels))
labels = ['Age', '+Gender', '+Income', '+Interests', '+All']


# Draw the bar chart.
plt.figure(figsize=(6.5, 2.5))
for idx, tt in enumerate(['orig', 'gender', 'income', 'interests', 'all']):
    conf = coefficients_dict[tt]
    plt.errorbar(x=idx, y=np.exp(conf['coefficients']), yerr=[[np.exp(conf['coefficients']) - np.exp(conf['lower'])], [np.exp(conf['upper']) - np.exp(conf['coefficients'])]], capsize=10, elinewidth=2, capthick=2, color='k')
    plt.bar(x=idx, height=np.exp(conf['coefficients']), color=five_colors[idx])
    print(np.exp(conf['coefficients']))
    # plt.text(x=1.35, y=idx, s='{:.2f} [{:.2f}, {:.2f}]'.format(np.exp(conf['coefficients']), np.exp(conf['lower']), np.exp(conf['upper'])))

# plt.fill_between(x=[0.9,1],y1=[-0.5,-0.5], y2=[4.5, 4.5], color='gray',alpha=0.1)
# plt.vlines(x=1, ymin=-0.5, ymax=4.5, color='gray', ls='--')
# # plt.gca().invert_yaxis()
# plt.xlim(0.9,1.5)
plt.xticks([0,1,2,3,4], [], fontsize=18)
plt.ylim(1,1.2)
plt.ylabel('Odds Ratio')
# plt.yticks([0,1,2,3,4],[])
plt.savefig('./figures/figure1/odds_ratio_2.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)


In [ ]:
coefficients_dict_OLS = {}
all_coefficients_dict_OLS = {}
X = dummmy_df[['age_conti']]
y = dummmy_df[['m_ratio']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.OLS(y, X).fit(cov_type='HC0')
print(model.summary())
# Extract coefficients and confidence intervals.
coefficients = model.params
conf = model.conf_int()
conf['coefficients'] = coefficients
conf.columns = ['lower', 'upper', 'coefficients']
coefficients_dict_OLS['orig'] = conf.loc['age_conti']
all_coefficients_dict_OLS['orig'] = conf

In [ ]:
X = dummmy_df[['age_conti','gender_F']]
y = dummmy_df[['m_ratio']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.OLS(y, X).fit(cov_type='HC0')
print(model.summary())
# Extract coefficients and confidence intervals.
coefficients = model.params
conf = model.conf_int()
conf['coefficients'] = coefficients
conf.columns = ['lower', 'upper', 'coefficients']
coefficients_dict_OLS['gender'] = conf.loc['age_conti']
all_coefficients_dict_OLS['gender'] = conf

In [ ]:
X = dummmy_df[['age_conti','mod_price_range_conti', 'fre_city_level_eng_first', 'fre_city_level_eng_fourth',
       'fre_city_level_eng_second', 'fre_city_level_eng_third',
       'fre_community_level_eng_Countryside',
       'fre_community_level_eng_County']]
y = dummmy_df[['m_ratio']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.OLS(y, X).fit(cov_type='HC0')
print(model.summary())
# Extract coefficients and confidence intervals.
coefficients = model.params
conf = model.conf_int()
conf['coefficients'] = coefficients
conf.columns = ['lower', 'upper', 'coefficients']
coefficients_dict_OLS['income'] = conf.loc['age_conti']
all_coefficients_dict_OLS['income'] = conf

In [ ]:
X = dummmy_df[['age_conti', 'entertainment_ratio',
       'news_ratio', 'watch_entropy_1st']]
y = dummmy_df[['m_ratio']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.OLS(y, X).fit(cov_type='HC0')
print(model.summary())
# Extract coefficients and confidence intervals.
coefficients = model.params
conf = model.conf_int()
conf['coefficients'] = coefficients
conf.columns = ['lower', 'upper', 'coefficients']
coefficients_dict_OLS['interests'] = conf.loc['age_conti']
all_coefficients_dict_OLS['interests'] = conf

In [ ]:
X = dummmy_df[['age_conti','gender_F',
               'mod_price_range_conti', 'fre_city_level_eng_first', 'fre_city_level_eng_fourth', 'fre_city_level_eng_second', 'fre_city_level_eng_third', 'fre_community_level_eng_Countryside',
               'fre_community_level_eng_County', 'entertainment_ratio', 'news_ratio', 'watch_entropy_1st']]
y = dummmy_df[['m_ratio']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.OLS(y, X).fit(cov_type='HC0')
print(model.summary())
# Extract coefficients and confidence intervals.
coefficients = model.params
conf = model.conf_int()
conf['coefficients'] = coefficients
conf.columns = ['lower', 'upper', 'coefficients']
coefficients_dict_OLS['all'] = conf.loc['age_conti']
all_coefficients_dict_OLS['all'] = conf

In [ ]:
(0.405- 0.396)/(1-0.405)

In [ ]:
labels = conf.index
positions = range(len(labels))

# Draw the bar chart.
plt.figure(figsize=(10, 6))
for idx, tt in enumerate(['orig', 'gender', 'income', 'interests', 'all']):
    conf = coefficients_dict_OLS[tt]
    plt.errorbar(y=idx, x= conf['coefficients'], xerr=[[conf['coefficients'] - conf['lower']], [conf['upper'] - conf['coefficients']]], capsize=5, elinewidth=2, capthick=2, fmt='o')

plt.fill_between(x=[0,0.01],y1=[-0.5,-0.5], y2=[4.5, 4.5], color='gray',alpha=0.1)
plt.vlines(x=0, ymin=-0.5, ymax=4.5, color='gray', ls='--')
plt.gca().invert_yaxis()
plt.xlim(-0.02,0.01)
plt.xticks([-0.02,-0.01, 0, 0.01])
plt.ylim(-0.5,4.5)
plt.xlabel('Effect Size')
plt.yticks([0,1,2,3,4],[])
plt.savefig('./figures/figure1/effect_size.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)




# plt.xticks(positions, labels)
# plt.axhline(0, color='red', linewidth=0.8, linestyle='--')
# plt.title('Coefficients with 95% Confidence Intervals')
# plt.xlabel('Variables')
# plt.ylabel('Effect Size')
# plt.tight_layout()
# plt.show()

In [ ]:
# ['18-23','24-30','31-40','41-50','51-60','60+'],
labels = conf.index
positions = range(len(labels))
five_colors = sns.color_palette("tab10",5)
x = np.array([i for i in np.linspace(0,1,np.arange(18,24).shape[0]+1)[:-1]] + [1+i for i in np.linspace(0,1,np.arange(24,31).shape[0]+1)[:-1]] \
+[2+i for i in np.linspace(0,1,np.arange(31,41).shape[0]+1)[:-1]] + [3+i for i in np.linspace(0,1,np.arange(41,51).shape[0]+1)[:-1]]\
+[4+i for i in np.linspace(0,1,np.arange(51,61).shape[0]+1)[:-1]] + [5.0])
x_show = np.arange(0,60-18+2)
labels = ['Age', '+Gender', '+Income', '+Interests', '+All']

def get_linear(x, b_0, b_1, others):
    return (b_0+b_1*x+others).reshape(1,-1)

def get_y_range(x, conf, dummmy_df):
    b_0 = conf.loc['const']
    b_1 = conf.loc['age_conti']
    other_columns = list(filter(lambda x: x not in ['const', 'age_conti'], list(conf.index)))
    other_values = 0 
    print(other_columns)
    for cc in other_columns:
        mean_values = dummmy_df[cc].mean()
        cc_avg = conf.loc[cc]['coefficients']*mean_values
        other_values += cc_avg
        
    l1 = get_linear(x, b_0['coefficients'], b_1['coefficients'], other_values)
    l2 = get_linear(x, b_0['lower'], b_1['lower'], other_values)
    l3 = get_linear(x, b_0['upper'], b_1['upper'], other_values)
    l4 = get_linear(x, b_0['lower'], b_1['upper'], other_values)
    l5 = get_linear(x, b_0['upper'], b_1['lower'], other_values)
    l_bound = np.vstack([l2, l3, l4, l5])
    l_upper = np.max(l_bound, axis=0,keepdims=True)
    l_lower = np.min(l_bound, axis=0,keepdims=True)
    return l1.reshape(-1), l_upper.reshape(-1), l_lower.reshape(-1)

    

# Draw the bar chart.
plt.figure(figsize=(6.5,6))
for idx, tt in enumerate(['orig', 'gender', 'income', 'interests', 'all']):
    conf = all_coefficients_dict_OLS[tt]

    l1, l_upper, l_lower = get_y_range(x, conf, dummmy_df)
    plt.plot(x_show, l1, color=five_colors[idx], lw=2, label=labels[idx], alpha=0.8)
    plt.fill_between(x=x_show , y1=l_lower, y2=l_upper, color=five_colors[idx], alpha=0.05)
    plt.xticks([i-18 for i in [18, 30, 40, 50, 60]], ['18', '30', '40', '50', '60'])


plt.hlines(y=dummmy_df['m_ratio'].mean(), xmin=0, xmax=60-18, lw=5, color='gray', ls='--', alpha=0.5,)
plt.legend(fontsize=20)
plt.xlabel('Age')
plt.ylabel('Depth of Deep Ics')
plt.savefig('./figures/figure1/effectsize_fitted.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
# plt.fill_between(x=[0.9,1],y1=[-0.5,-0.5], y2=[4.5, 4.5], color='gray',alpha=0.1)
# plt.vlines(x=1, ymin=-0.5, ymax=4.5, color='gray', ls='--')
# plt.gca().invert_yaxis()
# # plt.xlim(0.9,1.5)
# # plt.ylim(-0.5,4.5)
# plt.xlabel('Odds Ratio')
# plt.yticks([0,1,2,3,4],[])
# plt.savefig('./figures/figure1/odds_ratio.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

# plt.xticks(positions, labels)
# plt.axhline(0, color='red', linewidth=0.8, linestyle='--')
# plt.title('Coefficients with 95% Confidence Intervals')
# plt.xlabel('Variables')
# plt.ylabel('Effect Size')
# plt.tight_layout()
# plt.show()

In [ ]:
conf

In [ ]:
labels = conf.index
positions = range(len(labels))
five_colors = sns.color_palette("tab10",5)
labels = ['Age', '+Gender', '+Income', '+Interests', '+All']


# Draw the bar chart.
plt.figure(figsize=(6.5, 2.5))
for idx, tt in enumerate(['orig', 'gender', 'income', 'interests', 'all']):
    conf = coefficients_dict_OLS[tt]
    plt.errorbar(x=idx, y=conf['coefficients'], yerr=[[conf['coefficients'] - conf['lower']], [conf['upper'] - conf['coefficients']]], capsize=10, elinewidth=2, capthick=2, color='k')
    plt.bar(x=idx, height=conf['coefficients'],  color=five_colors[idx])

plt.xticks([0,1,2,3,4], [], rotation=-30)
plt.ylim(-0.015,0)
plt.ylabel('Reg. Coef.')
plt.savefig('./figures/figure1/effectsize_2.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
data_rec_p1 = pd.read_pickle('./avg_rec_entropy_p1.pkl')
data_rec_p2 = pd.read_pickle('./avg_rec_entropy_p1.pkl')
data_rec = pd.concat([data_rec_p1, data_rec_p2])
data_rec_p1 = 0
data_rec_p2 = 0

data_rec = data_rec.groupby('user_id')['avg_num_entropy'].mean().reset_index()



In [ ]:
dummmy_df =  dummmy_df.merge(data_rec, on='user_id')

In [ ]:
X = dummmy_df[['age_conti','gender_F', 'avg_num_entropy',
               'mod_price_range_conti', 'fre_city_level_eng_first', 'fre_city_level_eng_fourth', 'fre_city_level_eng_second', 'fre_city_level_eng_third', 'fre_community_level_eng_Countryside',
               'fre_community_level_eng_County', 'entertainment_ratio', 'news_ratio', 'watch_entropy_1st']]
y = dummmy_df[['indicator']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.Logit(y, X).fit(cov_type='HC0')
print(model.summary())
# Extract coefficients and confidence intervals.
coefficients = model.params
conf = model.conf_int()
conf['coefficients'] = coefficients
conf.columns = ['lower', 'upper', 'coefficients']

In [ ]:
X = dummmy_df[['age_conti','gender_F','avg_num_entropy',
               'mod_price_range_conti', 'fre_city_level_eng_first', 'fre_city_level_eng_fourth', 'fre_city_level_eng_second', 'fre_city_level_eng_third', 'fre_community_level_eng_Countryside',
               'fre_community_level_eng_County', 'entertainment_ratio', 'news_ratio', 'watch_entropy_1st']]
y = dummmy_df[['m_ratio']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.OLS(y, X).fit(cov_type='HC0')
print(model.summary())
# Extract coefficients and confidence intervals.
coefficients = model.params
conf = model.conf_int()
conf['coefficients'] = coefficients
conf.columns = ['lower', 'upper', 'coefficients']
coefficients_dict_OLS['all'] = conf.loc['age_conti']
all_coefficients_dict_OLS['all'] = conf

In [ ]:
X = dummmy_df[['age_conti']]
y = dummmy_df[['avg_num_entropy']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.OLS(y, X).fit(cov_type='HC0')
print(model.summary())

In [ ]:
X = dummmy_df[['age_conti','gender_F',
               'mod_price_range_conti', 'fre_city_level_eng_first', 'fre_city_level_eng_fourth', 'fre_city_level_eng_second', 'fre_city_level_eng_third', 'fre_community_level_eng_Countryside',
               'fre_community_level_eng_County', 'entertainment_ratio', 'news_ratio', 'watch_entropy_1st']]
y = dummmy_df[['avg_num_entropy']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.OLS(y, X).fit(cov_type='HC0')
print(model.summary())

In [ ]:
X = dummmy_df[['age_conti','gender_F']]
y = dummmy_df[['avg_num_entropy']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.OLS(y, X).fit(cov_type='HC0')
print(model.summary())

In [ ]:
X = dummmy_df[['age_conti',
               'mod_price_range_conti', 'fre_city_level_eng_first', 'fre_city_level_eng_fourth', 'fre_city_level_eng_second', 'fre_city_level_eng_third', 'fre_community_level_eng_Countryside',
               'fre_community_level_eng_County']]
y = dummmy_df[['avg_num_entropy']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.OLS(y, X).fit(cov_type='HC0')
print(model.summary())

In [ ]:
X = dummmy_df[['age_conti', 'entertainment_ratio', 'news_ratio', 'watch_entropy_1st']]
y = dummmy_df[['avg_num_entropy']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.OLS(y, X).fit(cov_type='HC0')
print(model.summary())

In [ ]:
X = dummmy_df[['age_conti']]
y = dummmy_df[['m_ratio']]

# Add an intercept.
X = sm.add_constant(X)

# Fit the regression model.
model = sm.OLS(y, X).fit(cov_type='HC0')
print(model.summary())